# مشروع بناء وكيل للإجابة عن أسئلة المدن السعودية

في هذا المشروع الختامي، ستقوم ببناء وكيل (Agent) ذكي باستخدام LangChain يمكنه الإجابة عن أسئلة حول المدن السعودية. سيتصل الوكيل بأداة (Tool) مخصصة تقرأ من مجموعة بيانات، ثم يستخدم نموذجاً لغوياً لتقديم إجابات مفيدة.

**الأهداف:**
- قراءة ومعالجة بيانات المدن السعودية.
- تعريف أداة خاصة تقوم باسترجاع المعلومات من DataFrame.
- دمج الأداة مع وكيل LangChain.
- تشغيل الوكيل للحصول على إجابات عن أسئلة المستخدم.
- (اختياري) إضافة أدوات متقدمة مثل المقارنة والذاكرة.

لنبدأ!

## 1. المتطلبات ومفتاح API

تأكد من تثبيت المكتبات المطلوبة:
```
pip install pandas langchain langchain-openai langgraph
```

ستحتاج إلى مفتاح API لخدمة DeepSeek. قم بتعيينه في الخلية التالية.

In [ ]:
import os

# ضع مفتاح API الخاص بك هنا
os.environ["DEEPSEEK_API_KEY"] = "ضع-مفتاحك-هنا"

## 2. استيراد المكتبات وقراءة البيانات

سنستورد المكتبات اللازمة ونقرأ ملف `saudi_cities.csv` الذي يحتوي على بيانات 10 مدن سعودية.

In [ ]:
import pandas as pd
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# قراءة ملف البيانات
df = pd.read_csv("saudi_cities.csv")

# عرض أول 5 صفوف للتأكد
df.head()

## 3. بناء الأداة الخاصة (Custom Tool)

سنقوم بإنشاء أداة تستقبل اسم المدينة وتعيد معلوماتها (عدد السكان والمنطقة). هذه الأداة ستكون دالة بايثون نمررها إلى الوكيل.

تلميح: استخدم `df[df['city'] == city]` لتصفية الصف المناسب، ثم استخرج القيم المطلوبة. تأكد من التعامل مع حالة عدم وجود المدينة.

In [ ]:
def city_info(city: str) -> str:
    """
    إرجاع معلومات عن مدينة سعودية (عدد السكان والمنطقة).
    المدخل: اسم المدينة باللغة الإنجليزية.
    المخرج: نص يحتوي على عدد السكان والمنطقة.
    """
    # TODO: اكتب الكود الذي يستعلم من df عن المدينة ويعيد النص المناسب.
    # في حالة عدم وجود المدينة، أرجع رسالة "لم يتم العثور على المدينة".
    ...

## 4. إعداد النموذج اللغوي (LLM)

سنستخدم نموذج DeepSeek من خلال `ChatOpenAI`. لاحظ أننا نمرر `api_key` من متغير البيئة الذي عينّاه سابقاً.

In [ ]:
llm = ChatOpenAI(
    model="deepseek-chat",
    base_url="https://api.deepseek.com/v1",
    api_key=os.environ["DEEPSEEK_API_KEY"]
)

## 5. إنشاء الوكيل (Create the Agent)

الآن سنقوم بدمج الأداة مع النموذج لإنشاء وكيل قادر على استخدامها. استخدم الدالة `create_agent` من LangChain.

In [ ]:
# TODO: أنشئ الوكيل باستخدام create_agent وامنحه الأداة التي بنيناها.
agent = ...

## 6. تشغيل الوكيل واختباره

لنجرب الوكيل بسؤال عن مدينة سعودية معروفة. استخدم `agent.invoke()` مع بنية الرسائل المناسبة، ثم اطبع الرد.

In [ ]:
# تجربة الوكيل بسؤال عن مدينة الرياض
response = agent.invoke({
    "messages": [{"role": "user", "content": "ما هو عدد سكان مدينة الرياض وفي أي منطقة تقع؟"}]
})
print(response["messages"][-1].text)

## 7. تمديدات إضافية (Optional)

### 7.1 أداة مقارنة بين مدينتين

يمكنك إضافة أداة جديدة `compare_cities(city1, city2)` تعيد مقارنة عدد السكان بين مدينتين. أضف الأداة وعدّل الوكيل.

In [ ]:
# TODO: اختر كتابة أداة مقارنة (دالة) وتضمينها في وكيل جديد.
def compare_cities(city1: str, city2: str) -> str:
    """
    مقارنة عدد السكان بين مدينتين سعوديتين.
    """
    # TODO: استخدم df لاستخراج عدد سكان كل مدينة وأعد جملة مقارنة.
    ...

# TODO: أنشئ وكيلاً جديداً يحتوي على الأداتين.
agent_with_compare = ...

### 7.2 إضافة ذاكرة (Memory)

لتذكر المحادثات السابقة، يمكنك إضافة `checkpointer`. استخدم `MemorySaver` من LangGraph ومررها إلى `create_agent`.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# TODO: أنشئ checkpointer وأضفه إلى وكيل جديد ليتمكن من تذكر المحادثة.
# تلميح: استخدم create_agent مع الوسيط checkpointer=... ثم مرر config عند الاستدعاء.
memory = ...
agent_with_memory = ...

## 8. خاتمة

أحسنت! لقد قمت ببناء وكيل ذكي يستطيع الإجابة عن أسئلة حول المدن السعودية باستخدام أدوات مخصصة. يمكنك التوسع في هذا المشروع بإضافة المزيد من الأدوات أو استخدام قاعدة بيانات حقيقية.

نأمل أن تكون قد استمتعت بالتجربة.